# 03 -- Signal pipeline

The decoupled steps and their permutations.

In [ ]:
import sys
sys.path.insert(0, "../src")          # run from examples/

from asdea_sensors import SensorDataset
from asdea_sensors.config import settings

DATA = r"C:\Users\ppala\Desktop\02_31MAY2026"   # folder with the .h5 files

In [ ]:
ds = SensorDataset(path=DATA, date_source="filename", load_mode="auto", verbose=True)
ds.verbose  = True      # internal prints, ShakerMakerResults style
ds.n_jobs   = 4
ds.parallel = True
ds.devices

## Read the raw signal

In [ ]:
sig = ds.MOF00135.signal(components="all", mode="auto", workers=4, remove_mean=False)
sig.acc_x, sig.time, sig.dt, sig.fs, sig.n, sig.axes

## Each step on its own

In [ ]:
b = sig.baseline(method="polynomial", components="all")
f = b.filter(fmin=0.1, fmax=24.9, engine="obspy", order=4, zerophase=True, components="all")
d = f.derive(method="trapezoid", remove_mean=True, components="all")
d.vel_x, d.disp_x

## Permutations of order

In [ ]:
# baseline then filter
a = ds.MOF00135.signal().baseline().filter(0.1, 24.9).derive()
# filter then baseline
b = ds.MOF00135.signal().filter(0.1, 24.9).baseline().derive()
# scipy engine instead of obspy
c2 = ds.MOF00135.signal().baseline().filter(0.5, 12.0, engine="scipy", order=4).derive()
# no baseline at all
d2 = ds.MOF00135.signal().filter(0.1, 24.9).derive()

## Resample before processing

In [ ]:
s100 = ds.MOF00135.signal().resample(dt=0.01).baseline().filter(0.1, 24.9).derive()